# Generative AI Month 2 — Task 3
## RAG with Unsloth Dynamic 4-bit Quantization

**Student:** Mehak Zahra  
**Domain:** Generative AI  
**Internship:** Arch Technologies

Run cells in order using a **T4 GPU** runtime.

In [ ]:
!nvidia-smi

In [ ]:
%pip install -q unsloth sentence-transformers pypdf

In [ ]:
import torch
import numpy as np
from sentence_transformers import SentenceTransformer
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
documents = [
    {'source': 'submission_guidelines.txt', 'text': '''Arch Technologies internship report must include a code explanation and relevant working screenshots. The completed Word report must be converted into PDF format. The report filename should follow the required Name_Domain_Month2 convention.'''.strip()},
    {'source': 'submission_process.txt', 'text': '''Completed internship reports are submitted to submissions.archtech@gmail.com. Students should also publish their work on LinkedIn and mention Arch Technologies. The submission should be complete and professionally structured.'''.strip()},
    {'source': 'student_projects.txt', 'text': '''Mehak Zahra is completing Month 2 programming and artificial-intelligence domain projects for the Arch Technologies internship.'''.strip()},
]

embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
texts = [item['text'] for item in documents]
document_embeddings = embedding_model.encode(texts, convert_to_numpy=True, normalize_embeddings=True)
print('Documents:', len(documents))
print('Embedding shape:', document_embeddings.shape)

In [ ]:
MODEL_NAME = 'unsloth/Llama-3.2-1B-Instruct-unsloth-bnb-4bit'
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)
tokenizer = get_chat_template(tokenizer, chat_template='llama-3.2')

print('Model loaded:', MODEL_NAME)
print('Quantization: Unsloth Dynamic 4-bit')
print(f'Allocated GPU memory: {torch.cuda.memory_allocated() / 1024**3:.2f} GB')

In [ ]:
def retrieve_document(question):
    query = embedding_model.encode([question], convert_to_numpy=True, normalize_embeddings=True)[0]
    scores = document_embeddings @ query
    index = int(np.argmax(scores))
    return {**documents[index], 'score': float(scores[index])}

def generate_rag_answer(question):
    item = retrieve_document(question)
    prompt = f'''Use only the DOCUMENT to answer the QUESTION. Give a short factual answer and finish with Source: {item['source']}.

DOCUMENT:
{item['text']}

QUESTION:
{question}
'''
    messages = [{'role': 'user', 'content': prompt}]
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors='pt', return_dict=True).to('cuda')
    input_length = inputs['input_ids'].shape[-1]
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=100, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    answer = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True).strip()
    return answer, item

In [ ]:
question = 'Which format should the completed Word report be converted into?'
answer, source = generate_rag_answer(question)

print('QUESTION:')
print(question)
print('\nRETRIEVED SOURCE:')
print(f"{source['source']} | Similarity: {source['score']:.4f}")
print('\nRAG ANSWER:')
print(answer)
print('\nMODEL:')
print(MODEL_NAME)
print('GPU:', torch.cuda.get_device_name(0))